# EGC Enzyme Tree

Reads `.a.ec.txt` files from `Metagenome_Samples`, builds the enzyme (EC number) tree, fetches EC names and hierarchy from KEGG BRITE, and writes one `.enzyme.json` per sample.

**Run order:** Cell by cell top to bottom. Restart kernel before re-running.

In [1]:
import json
import re
import time
import requests
from pathlib import Path
from collections import defaultdict
import sys
sys.path.append(r"C:\Users\sakoehn\Desktop\JGI\Notebooks")
from All_Coverages import allfold

## 1. Load sample list

In [2]:
with open(r'C:\Users\sakoehn\Desktop\JGI\sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f"Loaded {len(samples)} samples")

Loaded 128 samples


## 2. Read enzyme annotation files → enz_db

`enz_db[gene_id]` holds three parallel lists — one entry per observation of that gene:
- `lookupId`: the EC label from column 2 of the `.a.ec.txt` file
- `timepoint`: the sample this observation came from
- `scaffolds`: the scaffold this gene sits on

Both branches of the if/else write to the same `enz_db` parameter so no observations are lost.

In [3]:
def read_enzyme(directpath, filename, s, enz_db):
    """
    Parse one sample's .a.ec.txt and append each gene observation to enz_db.
    Both branches write to the enz_db parameter — no shadowing.
    """
    path = directpath / filename
    if not path.exists():
        print(f"  WARNING: {path} not found, skipping")
        return

    with open(path, "r") as f:
        for line in f:
            parts   = line.rstrip("\n").split("\t")
            gene_id = parts[0]
            label   = parts[2]           # EC label, e.g. "EC:3.6.1.23"
            scaff   = gene_id.rsplit('_', 2)[0]

            if gene_id not in enz_db:
                enz_db[gene_id] = {
                    'lookupId':  [label],
                    'timepoint': [s],
                    'scaffolds': [scaff],
                }
            else:
                enz_db[gene_id]['lookupId'].append(label)
                enz_db[gene_id]['timepoint'].append(s)
                enz_db[gene_id]['scaffolds'].append(scaff)


base   = Path(r"C:\Users\sakoehn\Desktop\JGI\Metagenome_Samples")
enz_db = {}

for s in samples:
    read_enzyme(base / s, f"{s}.a.ec.txt", s, enz_db)

print(f"enz_db: {len(enz_db):,} genes")

enz_db: 22,293,001 genes


## 2.5 Fetch and parse EC BRITE

Downloads `ko01000.keg` and builds a flat `brite` dict with depth, category, parent_id, and parent_name — the same structure used by the KEGG tree.

In [4]:
def normalize_ec(ec):
    """Strip 'EC:' prefix if present, e.g. 'EC:3.6.1.23' → '3.6.1.23'."""
    return ec[3:] if ec.startswith("EC:") else ec


url = "https://www.kegg.jp/kegg-bin/download_htext?htext=ko01000.keg&format=htext"
response = requests.get(url)
response.raise_for_status()
brite_file = response.text

brite = {}
current_at_depth = {}

for raw_line in brite_file.splitlines():
    line = raw_line.strip()
    if not line or line[0] not in "ABCDEFGHIJK":
        continue

    level_char = line[0]
    depth = ord(level_char) - ord('A') + 1

    # Strip HTML tags before matching (A-level lines: "A<b>1. Oxidoreductases</b>")
    rest = re.sub(r"<[^>]+>", "", line[1:]).strip()
    if not rest:
        continue

    # Optional trailing dot handles A-level format "1. Oxidoreductases"
    m = re.match(r"^(\d+(?:\.\d+){0,3})\.?\s+(.+)$", rest)
    if not m:
        continue

    node_id = m.group(1)
    name    = m.group(2).strip()

    parent      = current_at_depth.get(depth - 1)
    parent_id   = parent['id']   if parent else None
    parent_name = parent['name'] if parent else None
    category    = name if depth == 1 else current_at_depth.get(1, {}).get('name')

    current_at_depth[depth] = {'id': node_id, 'name': name}
    for d in list(current_at_depth.keys()):
        if d > depth:
            del current_at_depth[d]

    brite[node_id] = {
        'parent_id':   parent_id,
        'parent_name': parent_name,
        'name':        name,
        'category':    category,
        'depth':       depth,
    }

depth1 = {k: v for k, v in brite.items() if v['depth'] == 1}
print(f"Parsed {len(brite):,} BRITE entries  ({len(depth1)} depth-1 classes)")
for k, v in depth1.items():
    print(f"  {k}: {v['name']}")

Parsed 7,303 BRITE entries  (7 depth-1 classes)
  1: Oxidoreductases
  2: Transferases
  3: Hydrolases
  4: Lyases
  5: Isomerases
  6: Ligases
  7: Translocases


## 3. Build enzyme tree

For each gene in `enz_db`, strip the `EC:` prefix and look the bare EC number up in `brite`.  
Walk up the `parent_id` chain to the root, registering the gene and scaffold at **every ancestor node**.  
Only nodes with at least one observation from `enz_db` are added to the tree.

In [5]:
enz_tree = {}  # {ec_id: {name, category, depth, parent_id, parent_name, timepoints}}

start = time.time()

for gene_id, ginfo in enz_db.items():
    lookups    = ginfo.get('lookupId',  [])
    timepoints = ginfo.get('timepoint', [])
    scaffolds  = ginfo.get('scaffolds', [])

    for ec_raw, sample, scaff in zip(lookups, timepoints, scaffolds):
        ec_id = normalize_ec(ec_raw.split(';')[0].strip())

        if ec_id not in brite:
            continue

        # Walk from the EC up to the root, registering at every ancestor
        current = ec_id
        while current is not None:
            if current not in brite:
                break

            if current not in enz_tree:
                b = brite[current]
                enz_tree[current] = {
                    'name':        b['name'],
                    'category':    b['category'],
                    'depth':       b['depth'],
                    'parent_id':   b['parent_id'],
                    'parent_name': b['parent_name'],
                    'timepoints':  {},
                }

            node = enz_tree[current]

            if sample not in node['timepoints']:
                node['timepoints'][sample] = {
                    'egc':       0,
                    'norm_egc':  0.0,
                    'scaffolds': {},
                    'genes':     set(),
                }

            tp = node['timepoints'][sample]
            # Only count this gene once per (EC node, sample) — a multi-domain
            # protein can produce multiple rows for the same EC in .a.ec.txt.
            if gene_id not in tp['genes']:
                tp['scaffolds'][scaff] = tp['scaffolds'].get(scaff, 0) + 1
            tp['genes'].add(gene_id)

            current = brite[current]['parent_id']

print(f"Enzyme tree: {len(enz_tree):,} nodes  ({time.time() - start:.1f}s)")


Enzyme tree: 3,561 nodes  (93.5s)


## 4. Compute EGC and norm_egc

`egc = Σ (gene_count_on_scaffold × scaffold_coverage)`

`norm_egc = egc / Σ (gene_count_on_scaffold × scaffold_coverage) for ALL scaffolds`

The denominator is gene-weighted total depth so norm_egc is always in [0, 1].

In [6]:
# ── Gene-weighted depth denominator (corrected norm_egc) ────────────────────
# norm_egc should be in [0, 1]: fraction of all annotated gene copies in this
# sample that belong to this term.
#
# Old denominator: allfold[sample]['total'] = Σ(AvgFold for all scaffolds)
#   → can give norm_egc > 1 for annotations on high-coverage scaffolds
#
# New denominator: Σ(gene_count × AvgFold for all scaffolds in sample)
#   → EGC[term] ≤ this sum by construction, so norm_egc ≤ 1

from collections import defaultdict as _dd

# Count genes per scaffold per sample from enz_db
_gene_counts = _dd(lambda: _dd(int))
for _gid, _ginfo in enz_db.items():
    _scaff = _gid.rsplit('_', 2)[0]
    for _s in _ginfo.get('timepoint', []):
        _gene_counts[_s][_scaff] += 1

# weighted_total_per_sample[sample] = Σ(gene_count × AvgFold)
weighted_total_per_sample = {}
for _s, _counts in _gene_counts.items():
    _covs = allfold.get(_s, {})
    weighted_total_per_sample[_s] = sum(
        _counts.get(sc, 0) * _covs.get(sc, 0)
        for sc in _covs if sc != 'total'
    )

# Sanity check on first sample
_s0 = next(iter(weighted_total_per_sample))
_new = weighted_total_per_sample[_s0]
_old = allfold.get(_s0, {}).get('total', 0)
print(f'Sample {_s0}: old denom={_old:,.0f}  new denom={_new:,.0f}')
print(f'norm_egc values will be ~{_old/_new:.2f}x smaller, now bounded by 1.0')


Sample 3300060358: old denom=8,298,642  new denom=5,622,527
norm_egc values will be ~1.48x smaller, now bounded by 1.0


In [7]:
def avg_gene_map(node_id):
    node = enz_tree[node_id]
    for sample_name, data in node['timepoints'].items():
        scaffold_counts = data.get('scaffolds')
        if not scaffold_counts or sample_name not in allfold:
            continue

        covs = allfold[sample_name]
        sample_total_depth = weighted_total_per_sample.get(
            sample_name, allfold[sample_name]['total']
        )

        total_egc = 0
        for scaff, cnt in scaffold_counts.items():
            cov = covs.get(scaff, 0)
            total_egc += cnt * cov

        data['egc']      = total_egc
        data['norm_egc'] = (
            total_egc / sample_total_depth
            if sample_total_depth > 0 else 0.0
        )


for node_id in enz_tree:
    avg_gene_map(node_id)
print('EGC computation complete')


EGC computation complete


## 5. Write .enzyme.json files

In [8]:
base = Path(r"C:\Users\sakoehn\Desktop\JGI\DataStructures\Enzyme")
base.mkdir(parents=True, exist_ok=True)

for s in samples:
    tp_out = {}
    for ec_id, node in enz_tree.items():
        if s not in node['timepoints']:
            continue
        tp = node['timepoints'][s]
        tp_out[ec_id] = {
            'name':        node['name'],
            'depth':       node['depth'],
            'category':    node['category'],
            'parent_id':   node['parent_id'],
            'parent_name': node['parent_name'],
            'egc':         tp.get('egc',       0),
            'norm_egc':    tp.get('norm_egc',  0.0),
            'scaffolds':   tp.get('scaffolds', {}),
        }
    with open(base / f"{s}.enzyme.json", 'w') as f:
        json.dump(tp_out, f, indent=2)

print(f"Wrote {len(samples)} .enzyme.json files to {base}")

Wrote 128 .enzyme.json files to C:\Users\sakoehn\Desktop\JGI\DataStructures\Enzyme
